In [ ]:
from google.colab import drive
import pandas as pd
import json
import os
from datasets import load_dataset
from huggingface_hub import login

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
SAVE_PATH = "/content/drive/MyDrive/SentinelGPT_Project/"
os.makedirs(SAVE_PATH, exist_ok=True)

print("✅ Setup complete. Move to the next cell to input your token.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete. Move to the next cell to input your token.


In [ ]:
# This cell will wait for you.
from huggingface_hub import login

print("--- HUGGING FACE AUTHENTICATION ---")
print("1. Go to https://huggingface.co/settings/tokens")
print("2. Copy your 'READ' token.")
print("3. Paste it into the box below and press ENTER.")

# login() triggers an interactive input box that stops the execution
login()

print("\n✅ Token accepted! Now you can run the rest of the script.")

--- HUGGING FACE AUTHENTICATION ---
1. Go to https://huggingface.co/settings/tokens
2. Copy your 'READ' token.
3. Paste it into the box below and press ENTER.



✅ Token accepted! Now you can run the rest of the script.


In [ ]:
JSONL_INPUT = SAVE_PATH + "data/results_python_SC.jsonl"

def process_audit_logs(file_path):
    data = []
    if os.path.exists(file_path):
        print(f"Reading audit logs from {file_path}...")
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    # Label 1 = Attack
                    data.append({'text': item['question'], 'label': 1, 'source': 'gpt_store_audit'})
                except: continue
        return pd.DataFrame(data)
    else:
        print("⚠️ Warning: JSONL not found. Continuing with Open Source data only.")
        return pd.DataFrame(columns=['text', 'label', 'source'])

df_audit = process_audit_logs(JSONL_INPUT)
print(f"✅ Formatted {len(df_audit)} real-world attack samples.")

Reading audit logs from /content/drive/MyDrive/SentinelGPT_Project/data/results_python_SC.jsonl...
✅ Formatted 196 real-world attack samples.


In [ ]:
print("Fetching 100k Rows...")

# --- 1. MALICIOUS DATA ---
print("Loading HackAPrompt (Authenticated)...")
try:
    ds_hack = load_dataset("hackaprompt/hackaprompt-dataset", split="train")
    df_hack = ds_hack.to_pandas()[['user_input']].rename(columns={'user_input': 'text'})
    df_hack['label'] = 1
    df_hack['source'] = 'hackaprompt'
    # Target ~45,000 malicious samples
    df_hack = df_hack.sample(n=min(len(df_hack), 45000), random_state=42)
except Exception as e:
    print(f"❌ Error loading HackAPrompt: {e}")
    df_hack = pd.DataFrame()

print("Loading Wild Jailbreaks...")
ds_wild = load_dataset("TrustAIRLab/in-the-wild-jailbreak-prompts", "jailbreak_2023_12_25", split="train")
df_wild = ds_wild.to_pandas().rename(columns={'prompt': 'text'})
df_wild['label'] = 1
df_wild['source'] = 'trustairlab_wild'
df_wild = df_wild[['text', 'label', 'source']]

# --- 2. CLEAN DATA ---
print("Loading Alpaca Safe Samples...")
ds_alpaca = load_dataset("tatsu-lab/alpaca", split="train")
df_alpaca = ds_alpaca.to_pandas()
df_alpaca['text'] = df_alpaca['instruction'] + " " + df_alpaca['input']
df_alpaca = df_alpaca[['text']].copy()
df_alpaca['label'] = 0
df_alpaca['source'] = 'alpaca_clean'
# Target ~50,000 safe samples
df_alpaca = df_alpaca.sample(n=min(len(df_alpaca), 50000), random_state=42)

print("✅ Data download complete.")

Fetching 100k Rows...
Loading HackAPrompt (Authenticated)...


hackaprompt.parquet:   0%|          | 0.00/150M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/601757 [00:00<?, ? examples/s]

Loading Wild Jailbreaks...
Loading Alpaca Safe Samples...
✅ Data download complete.


In [ ]:
# 1. Combine
master_df = pd.concat([df_audit, df_hack, df_wild, df_alpaca], ignore_index=True)

# 2. Clean
master_df.dropna(subset=['text'], inplace=True)
master_df = master_df[master_df['text'].str.strip().astype(bool)]
master_df = master_df.drop_duplicates(subset=['text'])

# 3. Force EXACTLY 100,000 rows
target = 100000
current = len(master_df)

if current > target:
    master_df = master_df.sample(n=target, random_state=42)
elif current < target:
    diff = target - current
    print(f"Short by {diff} rows. Upsampling audit data...")
    upsample = df_audit.sample(n=diff, replace=True, random_state=42)
    master_df = pd.concat([master_df, upsample], ignore_index=True)

# 4. Final Shuffle
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Save and Summary
print(f"\n🔥 100k DATASET COMPLETE 🔥")
print(f"Final Count: {len(master_df)} rows")
print(f"Class Balance:\n{master_df['label'].value_counts()}")

master_df.to_pickle(SAVE_PATH + "sentinel_100k_master.pkl")
master_df.to_csv(SAVE_PATH + "sentinel_100k_master.csv", index=False)

print(f"\n💾 MASTER DATASET SAVED TO GOOGLE DRIVE!")

Short by 9704 rows. Upsampling audit data...

🔥 100k DATASET COMPLETE 🔥
Final Count: 100000 rows
Class Balance:
label
0    50000
1    50000
Name: count, dtype: int64

💾 MASTER DATASET SAVED TO GOOGLE DRIVE!
